In [ ]:
# ✅ Install requirements
!pip install librosa PyWavelets scikit-learn torch torchvision --quiet

# ✅ Clone dataset
!rm -rf /content/donateacry-corpus
!git clone https://github.com/gveres/donateacry-corpus.git

# ✅ Check structure
!ls donateacry-corpus/wav


Cloning into 'donateacry-corpus'...
remote: Enumerating objects: 1616, done.
remote: Total 1616 (delta 0), reused 0 (delta 0), pack-reused 1616 (from 1)
Receiving objects: 100% (1616/1616), 67.06 MiB | 15.45 MiB/s, done.
Resolving deltas: 100% (43/43), done.
ls: cannot access 'donateacry-corpus/wav': No such file or directory


In [ ]:
# ✅ Install dependencies (run this once in Colab/Notebook)
!pip install librosa PyWavelets noisereduce --quiet

import os, shutil, gc
import librosa, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import pywt
import noisereduce as nr
from scipy.signal import butter, lfilter

# ✅ Matplotlib non-interactive backend
import matplotlib
matplotlib.use('Agg')

# ✅ Paths
BASE_AUDIO_DIR = '/content/donateacry-corpus/donateacry_corpus_cleaned_and_updated_data'
OUTPUT_IMG_DIR = '/content/cry_images'

# ✅ RAM-optimized parameters
SAMPLE_RATE = 8000        # reduced sample rate
DURATION = 0.5            # shorter clip duration (0.5 sec)
OVERLAP = 0.25            # less overlap
SCALES = np.arange(1, 64) # fewer CWT scales
FIGSIZE = (2, 2)          # smaller output image

# -----------------------------
# 🔹 Bandpass Filter
# -----------------------------
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut=200, highcut=2000, fs=8000, order=5):
    """Keep only baby cry relevant frequencies (200 Hz - 2 kHz)."""
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y

# -----------------------------
# 🔹 Scalogram Generator
# -----------------------------
def generate_scalogram(y, sr, out_file):
    coef, _ = pywt.cwt(y, SCALES, 'morl', sampling_period=1/sr)
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.imshow(np.abs(coef), extent=[0, 1, 1, len(SCALES)], cmap='jet',
              aspect='auto', vmax=np.abs(coef).max())
    ax.axis('off')
    fig.tight_layout(pad=0)

    temp_png = out_file.replace('.jpg', '_temp.png')
    fig.savefig(temp_png, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    img = Image.open(temp_png).convert("RGB")
    img.save(out_file, format='JPEG', quality=85)
    os.remove(temp_png)
    gc.collect()

# -----------------------------
# 🔹 Segmentation + Processing
# -----------------------------
def segment_and_convert():
    if os.path.exists(OUTPUT_IMG_DIR):
        shutil.rmtree(OUTPUT_IMG_DIR)
    os.makedirs(OUTPUT_IMG_DIR)

    for cls in os.listdir(BASE_AUDIO_DIR):
        cls_path = os.path.join(BASE_AUDIO_DIR, cls)
        if not os.path.isdir(cls_path) or cls.startswith('.'):
            continue

        output_cls = os.path.join(OUTPUT_IMG_DIR, cls)
        os.makedirs(output_cls, exist_ok=True)

        for file in tqdm(os.listdir(cls_path), desc=f"Processing {cls}"):
            if not file.lower().endswith('.wav'):
                continue

            path = os.path.join(cls_path, file)
            try:
                # Load audio
                y, sr = librosa.load(path, sr=SAMPLE_RATE)

                # ✅ Step 1: Noise reduction
                y = nr.reduce_noise(y=y, sr=sr)

                # ✅ Step 2: Bandpass filter (keep 200 Hz – 2 kHz)
                y = bandpass_filter(y, lowcut=200, highcut=2000, fs=sr)

                # ✅ Step 3: Segment & generate scalograms
                segment_len = int(sr * DURATION)
                step = int(segment_len * (1 - OVERLAP))

                for i in range(0, len(y) - segment_len, step):
                    clip = y[i:i + segment_len]
                    out_file = os.path.join(output_cls, f"{file}_{i}.jpg")
                    if os.path.exists(out_file):
                        continue
                    generate_scalogram(clip, sr, out_file)

            except Exception as e:
                print(f"❌ Error processing {file}: {e}")
            finally:
                gc.collect()

# ✅ Run preprocessing
segment_and_convert()


Processing discomfort: 100%|██████████| 27/27 [02:09<00:00,  4.80s/it]


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/content/cry_images"
BATCH_SIZE = 32
LR = 3e-4
EPOCHS = 50


In [ ]:
class SimCLRTransform:
    def __init__(self):
        self.base_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.3),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)


In [ ]:
class SimCLRDataset(Dataset):
    def __init__(self, root, transform):
        self.dataset = datasets.ImageFolder(root=root, transform=transform)

    def __getitem__(self, idx):
        (view1, view2), label = self.dataset[idx]
        return view1, view2

    def __len__(self):
        return len(self.dataset)


In [ ]:
def nt_xent_loss(z_i, z_j, temperature=0.5):
    z_i = nn.functional.normalize(z_i, dim=1)
    z_j = nn.functional.normalize(z_j, dim=1)
    N = z_i.size(0)

    representations = torch.cat([z_i, z_j], dim=0)
    similarity_matrix = torch.matmul(representations, representations.T)

    # remove self-similarity
    mask = torch.eye(2*N, dtype=torch.bool).to(z_i.device)
    similarity_matrix = similarity_matrix[~mask].view(2*N, 2*N-1)

    positives = torch.sum(z_i * z_j, dim=-1) / temperature
    positives = torch.cat([positives, positives], dim=0)

    logits = similarity_matrix / temperature
    labels = torch.arange(N).repeat(2).to(z_i.device)

    loss = nn.CrossEntropyLoss()(logits, labels)
    return loss


In [ ]:
class SimCLR(nn.Module):
    def __init__(self, base_encoder=models.resnet18, out_dim=128):
        super().__init__()
        self.encoder = base_encoder(pretrained=False)
        num_ftrs = self.encoder.fc.in_features
        self.encoder.fc = nn.Identity()

        self.projector = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, z


In [ ]:
def train_simclr():
    dataset = SimCLRDataset(DATA_DIR, transform=SimCLRTransform())
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    model = SimCLR().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS+1):
        total_loss = 0
        model.train()

        for x_i, x_j in loader:
            x_i, x_j = x_i.to(DEVICE), x_j.to(DEVICE)

            _, z_i = model(x_i)
            _, z_j = model(x_j)

            loss = nt_xent_loss(z_i, z_j, temperature=0.5)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch}/{EPOCHS}, Loss={avg_loss:.4f}")

    torch.save(model.encoder.state_dict(), "simclr_encoder.pth")
    print("✅ Pretraining done. Encoder saved as simclr_encoder.pth")

train_simclr()


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 1/50, Loss=4.0929
Epoch 2/50, Loss=3.9744
Epoch 3/50, Loss=3.8685
Epoch 4/50, Loss=3.7527
Epoch 5/50, Loss=3.6784
Epoch 6/50, Loss=3.6522
Epoch 7/50, Loss=3.6230
Epoch 8/50, Loss=3.6012
Epoch 9/50, Loss=3.5848
Epoch 10/50, Loss=3.5782
Epoch 11/50, Loss=3.5658
Epoch 12/50, Loss=3.5529
Epoch 13/50, Loss=3.5509
Epoch 14/50, Loss=3.5391
Epoch 15/50, Loss=3.5396
Epoch 16/50, Loss=3.5276
Epoch 17/50, Loss=3.5200
Epoch 18/50, Loss=3.5151
Epoch 19/50, Loss=3.5078
Epoch 20/50, Loss=3.5080
Epoch 21/50, Loss=3.5021
Epoch 22/50, Loss=3.4977
Epoch 23/50, Loss=3.4937
Epoch 24/50, Loss=3.4895
Epoch 25/50, Loss=3.4802
Epoch 26/50, Loss=3.4845
Epoch 27/50, Loss=3.4814
Epoch 28/50, Loss=3.4827
Epoch 29/50, Loss=3.4723
Epoch 30/50, Loss=3.4764
Epoch 31/50, Loss=3.4742
Epoch 32/50, Loss=3.4714
Epoch 33/50, Loss=3.4638
Epoch 34/50, Loss=3.4568
Epoch 35/50, Loss=3.4603
Epoch 36/50, Loss=3.4597
Epoch 37/50, Loss=3.4591
Epoch 38/50, Loss=3.4601
Epoch 39/50, Loss=3.4566
Epoch 40/50, Loss=3.4573
Epoch 41/

In [ ]:
# ✅ 5-Fold Cross Validation with Frozen SimCLR Encoder
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold
import numpy as np

# -----------------------------
# 🔹 Config
# -----------------------------
DATA_DIR = "/content/cry_images"
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
N_SPLITS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 🔹 Transform
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

# -----------------------------
# 🔹 Load Pretrained Encoder
# -----------------------------
backbone = models.resnet18(weights=None)
in_dim = backbone.fc.in_features
backbone.fc = nn.Identity()
backbone.load_state_dict(torch.load("simclr_encoder.pth", map_location=DEVICE))
backbone.to(DEVICE)
backbone.eval()

# Freeze encoder
for param in backbone.parameters():
    param.requires_grad = False

# -----------------------------
# 🔹 CV Loop
# -----------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
accuracies = []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(dataset)))):
    print(f"\n🔹 Fold {fold+1}/{N_SPLITS}")
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE)

    # Classifier head
    classifier = nn.Linear(in_dim, len(dataset.classes)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(classifier.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        classifier.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.no_grad():
                h = backbone(x)
            logits = classifier(h)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # ✅ Validation
    classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            h = backbone(x)
            logits = classifier(h)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    acc = correct / total
    accuracies.append(acc)
    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

print("\n✅ Final 5-Fold Accuracy:", np.mean(accuracies))



🔹 Fold 1/5
Fold 1 Accuracy: 0.8262

🔹 Fold 2/5
Fold 2 Accuracy: 0.8367

🔹 Fold 3/5
Fold 3 Accuracy: 0.8429

🔹 Fold 4/5
Fold 4 Accuracy: 0.8218

🔹 Fold 5/5
Fold 5 Accuracy: 0.8514

✅ Final 5-Fold Accuracy: 0.8358192535933441


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
import numpy as np

# -----------------------------
# Config
# -----------------------------
DATA_DIR = "/content/cry_images"
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4
N_SPLITS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Transform + Augmentation
# -----------------------------
transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
labels = [y for _, y in dataset.samples]

# -----------------------------
# Load Pretrained ResNet-18 Encoder
# -----------------------------
backbone = models.resnet18(weights=None)
in_dim = backbone.fc.in_features
backbone.fc = nn.Identity()
backbone.load_state_dict(torch.load("simclr_encoder.pth", map_location=DEVICE))
backbone.to(DEVICE)

# Optional: fine-tune last block
for name, param in backbone.named_parameters():
    param.requires_grad = False
for name, param in backbone.named_parameters():
    if "layer4" in name:  # fine-tune last block
        param.requires_grad = True

# -----------------------------
# 5-Fold Stratified CV
# -----------------------------
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
accuracies = []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(dataset)), labels)):
    print(f"\n🔹 Fold {fold+1}/{N_SPLITS}")
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE)

    # MLP Classifier head
    classifier = nn.Sequential(
        nn.Linear(in_dim, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, len(dataset.classes))
    ).to(DEVICE)

    # Optimizer: classifier + fine-tuned backbone params
    optimizer = optim.Adam(
        list(classifier.parameters()) +
        [p for p in backbone.parameters() if p.requires_grad],
        lr=LR, weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    criterion = nn.CrossEntropyLoss()
    epoch_accuracies = []

    for epoch in range(EPOCHS):
        # -------- Training --------
        backbone.train()
        classifier.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            h = backbone(x)
            logits = classifier(h)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(backbone.parameters()) + list(classifier.parameters()), 1.0
            )
            optimizer.step()
            running_loss += loss.item() * x.size(0)

        scheduler.step(running_loss / len(train_loader.dataset))

        # -------- Validation --------
        backbone.eval()
        classifier.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                h = backbone(x)
                logits = classifier(h)
                preds = torch.argmax(logits, dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)

        epoch_acc = correct / total
        epoch_accuracies.append(epoch_acc)
        print(f"Epoch {epoch+1}/{EPOCHS}, Accuracy: {epoch_acc:.4f}")

    accuracies.append(epoch_accuracies)

# -----------------------------
# Summary
# -----------------------------
accuracies = np.array(accuracies)
for fold in range(N_SPLITS):
    print(f"\nFold {fold+1} Epoch-wise Accuracy:")
    print(accuracies[fold])

print("\n✅ Final 5-Fold Accuracy (last epoch):", np.mean(accuracies[:, -1]))



🔹 Fold 1/5
Epoch 1/5, Accuracy: 0.8355
Epoch 2/5, Accuracy: 0.8355
Epoch 3/5, Accuracy: 0.8355
Epoch 4/5, Accuracy: 0.8355
Epoch 5/5, Accuracy: 0.8355

🔹 Fold 2/5
Epoch 1/5, Accuracy: 0.8361
Epoch 2/5, Accuracy: 0.8361
Epoch 3/5, Accuracy: 0.8361
Epoch 4/5, Accuracy: 0.8361
Epoch 5/5, Accuracy: 0.8361

🔹 Fold 3/5
Epoch 1/5, Accuracy: 0.8361
Epoch 2/5, Accuracy: 0.8361
Epoch 3/5, Accuracy: 0.8361
Epoch 4/5, Accuracy: 0.8361
Epoch 5/5, Accuracy: 0.8361

🔹 Fold 4/5
Epoch 1/5, Accuracy: 0.8360
Epoch 2/5, Accuracy: 0.8360
Epoch 3/5, Accuracy: 0.8360
Epoch 4/5, Accuracy: 0.8360
Epoch 5/5, Accuracy: 0.8360

🔹 Fold 5/5
Epoch 1/5, Accuracy: 0.8360
Epoch 2/5, Accuracy: 0.8360
Epoch 3/5, Accuracy: 0.8360
Epoch 4/5, Accuracy: 0.8360
Epoch 5/5, Accuracy: 0.8360

Fold 1 Epoch-wise Accuracy:
[0.83548983 0.83548983 0.83548983 0.83548983 0.83548983]

Fold 2 Epoch-wise Accuracy:
[0.83610598 0.83610598 0.83610598 0.83610598 0.83610598]

Fold 3 Epoch-wise Accuracy:
[0.83610598 0.83610598 0.83610598 0.836